# Research Workflow with Memory (LangGraph)

This notebook demonstrates a LangGraph workflow that accumulates research notes over multiple steps and persists state across runs.


## What You Will Build

1. A step-by-step research workflow
2. Persistent memory with LangGraph `MemorySaver`
3. Multi-step note accumulation in one thread


## Memory and Persistence Explained

LangGraph persistence uses a checkpointer plus a `thread_id`.

- `checkpointer=MemorySaver()` stores state snapshots
- same `thread_id` means later calls can continue from earlier saved state
- we read last state via `app.get_state(config)` and continue building notes

This is how long-running research sessions can be resumed without losing context.


## Workflow Graph

```text
START
  -> collect_evidence
  -> synthesize_step_note
  -> persist_research_memory
  -> END
```


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import List, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define Research State

In [3]:
class ResearchState(TypedDict):
    current_question: str
    evidence_items: List[str]
    step_note: str
    research_notes: List[str]
    running_summary: str

print('State schema initialized')


State schema initialized


## Step 3 - Tiny Knowledge Base (Demo Data)

In [4]:
KNOWLEDGE_BASE = {
    'battery': [
        'Lithium-ion batteries generally degrade with heat and deep discharge cycles.',
        'Battery management systems can extend lifespan by reducing peak charge stress.',
        'Fast charging convenience can trade off with long-term cycle retention.',
    ],
    'supply chain': [
        'Single-source suppliers increase disruption risk during geopolitical shocks.',
        'Dual-sourcing usually improves resilience but can increase procurement complexity.',
        'Inventory buffers reduce stockout risk but increase carrying costs.',
    ],
    'pricing': [
        'Value-based pricing aligns revenue to customer outcomes.',
        'Cost-plus pricing is simple but may miss willingness-to-pay opportunities.',
        'Tiered pricing can improve conversion across customer segments.',
    ],
}

print('Knowledge base topics:', list(KNOWLEDGE_BASE.keys()))


Knowledge base topics: ['battery', 'supply chain', 'pricing']


## Step 4 - Node 1: Collect Evidence

In [5]:
def collect_evidence(state: ResearchState) -> dict:
    q = state['current_question'].lower()
    evidence = []

    if 'battery' in q:
        evidence.extend(KNOWLEDGE_BASE['battery'])
    if 'supply' in q or 'vendor' in q:
        evidence.extend(KNOWLEDGE_BASE['supply chain'])
    if 'price' in q or 'pricing' in q:
        evidence.extend(KNOWLEDGE_BASE['pricing'])

    if not evidence:
        evidence.append('No direct topic match found; capture as open research item for manual follow-up.')

    print('collect_evidence complete with', len(evidence), 'items')
    return {'evidence_items': evidence}


## Step 5 - Node 2: Synthesize Step Note

In [6]:
def synthesize_step_note(state: ResearchState) -> dict:
    note_lines = []
    note_lines.append('Question: ' + state['current_question'])
    note_lines.append('Key evidence:')
    for item in state['evidence_items'][:3]:
        note_lines.append('- ' + item)

    note = '\n'.join(note_lines)
    print('synthesize_step_note complete')
    return {'step_note': note}


## Step 6 - Node 3: Persist Research Memory

In [7]:
def persist_research_memory(state: ResearchState) -> dict:
    notes = list(state.get('research_notes', []))
    notes.append(state['step_note'])

    summary_lines = ['Running Research Summary']
    summary_lines.append(f'Total notes: {len(notes)}')
    if notes:
        summary_lines.append('Latest note heading: ' + notes[-1].split('\n')[0])

    summary = '\n'.join(summary_lines)
    print('persist_research_memory complete; notes now', len(notes))
    return {'research_notes': notes, 'running_summary': summary}


## Step 7 - Build Graph with Persistent Checkpointer

In [8]:
builder = StateGraph(ResearchState)
builder.add_node('collect_evidence', collect_evidence)
builder.add_node('synthesize_step_note', synthesize_step_note)
builder.add_node('persist_research_memory', persist_research_memory)

builder.add_edge(START, 'collect_evidence')
builder.add_edge('collect_evidence', 'synthesize_step_note')
builder.add_edge('synthesize_step_note', 'persist_research_memory')
builder.add_edge('persist_research_memory', END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)

print('Graph compiled with MemorySaver')


Graph compiled with MemorySaver


## Multi-Step Persistence Demo

We run multiple research questions under the same `thread_id`.
Before each step, we load previous saved state with `get_state(config)`.
This simulates a persistent research workspace that grows over time.


## Step 8 - Run Step 1

In [9]:
config = {'configurable': {'thread_id': 'research-memory-38-thread-1'}}

state1 = {
    'current_question': 'How do battery lifecycle constraints affect product planning?',
    'evidence_items': [],
    'step_note': '',
    'research_notes': [],
    'running_summary': '',
}

result1 = app.invoke(state1, config)
print('Step 1 complete; notes stored:', len(result1['research_notes']))


collect_evidence complete with 3 items
synthesize_step_note complete
persist_research_memory complete; notes now 1
Step 1 complete; notes stored: 1


## Step 9 - Run Step 2 (Resume from Persistent State)

In [10]:
previous = app.get_state(config).values
state2 = dict(previous)
state2['current_question'] = 'What supply chain tactics reduce vendor disruption risk?'

result2 = app.invoke(state2, config)
print('Step 2 complete; notes stored:', len(result2['research_notes']))


collect_evidence complete with 3 items
synthesize_step_note complete
persist_research_memory complete; notes now 2
Step 2 complete; notes stored: 2


## Step 10 - Run Step 3 (Resume Again)

In [11]:
previous = app.get_state(config).values
state3 = dict(previous)
state3['current_question'] = 'Which pricing strategy balances adoption and margin?'

result3 = app.invoke(state3, config)
print('Step 3 complete; notes stored:', len(result3['research_notes']))


collect_evidence complete with 3 items
synthesize_step_note complete
persist_research_memory complete; notes now 3
Step 3 complete; notes stored: 3


## Step 11 - Inspect Accumulated Memory

In [12]:
final_state = app.get_state(config).values
print(final_state['running_summary'])
print('\nAll accumulated notes:')
for i, note in enumerate(final_state['research_notes'], start=1):
    print('\n' + '=' * 80)
    print(f'Note {i}')
    print(note)


Running Research Summary
Total notes: 3
Latest note heading: Question: Which pricing strategy balances adoption and margin?

All accumulated notes:

Note 1
Question: How do battery lifecycle constraints affect product planning?
Key evidence:
- Lithium-ion batteries generally degrade with heat and deep discharge cycles.
- Battery management systems can extend lifespan by reducing peak charge stress.
- Fast charging convenience can trade off with long-term cycle retention.

Note 2
Question: What supply chain tactics reduce vendor disruption risk?
Key evidence:
- Single-source suppliers increase disruption risk during geopolitical shocks.
- Dual-sourcing usually improves resilience but can increase procurement complexity.
- Inventory buffers reduce stockout risk but increase carrying costs.

Note 3
Question: Which pricing strategy balances adoption and margin?
Key evidence:
- Value-based pricing aligns revenue to customer outcomes.
- Cost-plus pricing is simple but may miss willingness-to

## Step 12 - Inspect Checkpoint History

In [13]:
history = list(app.get_state_history(config))
print('Checkpoint count:', len(history))
for i, snapshot in enumerate(history):
    src = 'init'
    if snapshot.metadata and 'source' in snapshot.metadata:
        src = snapshot.metadata['source']
    keys = [k for k, v in snapshot.values.items() if v not in ('', [], None)]
    print(f'Checkpoint {i}: source={src}, populated_keys={keys}')


Checkpoint count: 15
Checkpoint 0: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 1: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 2: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 3: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 4: source=input, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 5: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 6: source=loop, populated_keys=['current_question', 'evidence_items', 'step_note', 'research_notes', 'running_summary']
Checkpoint 7: source=loop, populated_keys=['current_question'

## Key Takeaways

1. Persistent memory in LangGraph comes from checkpointer + stable thread id.
2. Each step can read prior saved state and append new research notes.
3. Checkpoint history gives an auditable timeline of how knowledge evolved.
